# CAWOT-CM V0 — End-to-end Colab demo

Pipeline:
1. Extract CLIP embeddings of the training pool
2. FAISS k-means clustering
3. Two coresets: **Random** (baseline) vs **V0** (farthest-from-centroid)
4. Fine-tune CLIP on each coreset
5. Evaluate text→image retrieval (R@1/5/10, mAP)

**Two run modes** (pick one in step 4):
- **Sanity mode** — generate dummy data, verify pipeline runs end-to-end (~5-10 min on T4 free)
- **PAB mode** — download N image zips from [TruongVox/Cawot-dataset](https://huggingface.co/datasets/TruongVox/Cawot-dataset), extract flat to `pab_data/images/`, run the pipeline. Start with `NUM_ZIPS=2-3` to fit Colab disk.

Recommended: run sanity mode first to make sure everything works, then switch to PAB.

## 1. Check GPU + disk

In [ ]:
!nvidia-smi -L
!df -h /content | tail -1

## 2. Get the code

Clone the private repo (you'll need a Personal Access Token with `repo` scope).
Replace `<your_pat>` and `<your_username>` below. Alternatively, upload the folder via Files panel and `%cd` into it.

In [ ]:
import os

# Option A — clone (private repo, needs PAT)
# !git clone https://<your_username>:<your_pat>@github.com/HohoHocCode/cawot-cm.git /content/cawot-cm
# PROJECT_DIR = "/content/cawot-cm"

# Option B — already uploaded via Files panel
PROJECT_DIR = "/content/cawot-cm-v0"

assert os.path.exists(PROJECT_DIR), f"Code not found at {PROJECT_DIR}. Upload folder or clone first."
os.chdir(PROJECT_DIR)
!ls

## 3. Install dependencies (~2 min)

In [ ]:
!pip install -q open_clip_torch faiss-gpu-cu12 einops pyyaml wandb tqdm huggingface_hub

## 4. Choose run mode

Set `MODE` to `"sanity"` or `"pab"`.

For PAB:
- Each zip is ~1.4 GB → 2 zips ≈ 2.8 GB download + extracted. Colab free disk fits ~50 GB.
- Start with `NUM_ZIPS = 2-3` to verify the flow; bump later.
- The script also fetches `train.json`, `test_query.json`, `test_gallery.json` from the same HF dataset. If those files are not yet on HF, the script will print a warning — upload them or pass them in manually.

In [ ]:
MODE = "sanity"   # "sanity" | "pab"
NUM_ZIPS = 2       # PAB only: how many imgs_*.zip to download

if MODE == "pab":
    DATA_ROOT = "/content/pab_data"
    SUBSET_SIZE = None   # use train_local.json after filtering
    K_CLUSTERS = 200
    # Download + extract
    !python scripts/download_pab.py --root {DATA_ROOT} --num-zips {NUM_ZIPS} --delete-zips-after-extract
    # Filter annotations to entries with locally available images
    !python scripts/filter_annotations.py --root {DATA_ROOT}
    TRAIN_JSON = "train_local.json"
    GALLERY_JSON = "test_gallery_local.json"
elif MODE == "sanity":
    DATA_ROOT = "/content/cawot-cm-v0/dummy_pab"
    SUBSET_SIZE = None
    K_CLUSTERS = 50
    !python scripts/make_dummy_data.py --root {DATA_ROOT} --n-train 2000 --n-query 100 --n-gallery 500
    TRAIN_JSON = "train.json"
    GALLERY_JSON = "test_gallery.json"
else:
    raise ValueError(MODE)

print(f"DATA_ROOT = {DATA_ROOT}")
print(f"TRAIN_JSON = {TRAIN_JSON}")
print(f"GALLERY_JSON = {GALLERY_JSON}")

## 5. Patch config.yaml with the chosen settings

In [ ]:
import yaml

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["root"] = DATA_ROOT
cfg["data"]["train_json"] = TRAIN_JSON
cfg["data"]["test_gallery_json"] = GALLERY_JSON
cfg["data"]["subset_size"] = SUBSET_SIZE
cfg["cluster"]["k"] = K_CLUSTERS

if MODE == "sanity":
    cfg["train"]["num_epochs"] = 1
    cfg["train"]["batch_size"] = 64
    cfg["embed"]["batch_size"] = 128
    cfg["eval"]["batch_size"] = 128
    cfg["data"]["num_workers"] = 2
    cfg["coreset"]["budget_ratio"] = 0.2
elif MODE == "pab":
    cfg["train"]["num_epochs"] = 3
    cfg["train"]["batch_size"] = 96
    cfg["embed"]["batch_size"] = 256
    cfg["eval"]["batch_size"] = 256
    cfg["data"]["num_workers"] = 2

with open("config.yaml", "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(yaml.dump(cfg, sort_keys=False))

## 6. Run end-to-end

Sanity mode: ~5-10 min on T4.
PAB mode (2 zips, ~13k train images): ~30-60 min depending on subset choices.

Steps are cached — safe to interrupt and re-run.

In [ ]:
!python scripts/run_v0.py --config config.yaml

## 7. Inspect results

In [ ]:
import json, pandas as pd
with open("outputs/eval/summary.json") as f:
    summary = json.load(f)
df = pd.DataFrame(summary).T
df

## 8. Interpreting results

**Sanity mode**: the absolute numbers don't mean anything — data is random. What matters is that the pipeline runs end-to-end with no errors and produces a `summary.json`. If Random and V0 give roughly similar metrics here, that's fine — there's no real signal in dummy data.

**PAB mode at full 1M, 20% budget** (target expectation, NOT the 2-zip Colab subset):

| Method | Expected R@1 |
|--------|-------------|
| Random | ~78-79% |
| V0     | ~79-80% (+0.5-1.5%) |

**Caveats**:
- V0 uses **CLIP + last-4-layer finetune + InfoNCE**, not IRRA. Absolute numbers will be lower than the CMP paper. What matters is the **relative** ranking V0 > Random.
- At 2-3 zips (~13-20k train images), numbers will be much lower — but the V0 > Random gap should still appear.
- If V0 ≤ Random on real PAB at scale → there's a bug in selection logic OR `K_CLUSTERS` is wrong scale (rule of thumb: K ≈ √(N/2)).

## 9. Next steps

1. **Sanity pass** → confirm pipeline runs end-to-end.
2. **PAB 2-3 zips** → confirm pipeline runs on real images, V0 > Random gap exists.
3. **ThunderCompute full 1M** → final numbers.
4. **Start V1**: cross-modal cost + submodlib facility-location.